# Plant Disease Classification - 3-Way Split (Train / Test / Validate)
Comparing ResNet50, EfficientNet-B0, and MobileNetV3Large on Guava and Citrus crops.

**Split:** 70% Train / 15% Validation / 15% Test (stratified)

In [2]:
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_ROOT = 'dataset'
OUTPUT_DIR = 'outputs_test3'
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 0.0001

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"TensorFlow: {tf.__version__}")

ModuleNotFoundError: No module named 'tensorflow'

## 1. Dataset Loading & Class Distribution

In [ ]:
crops = [d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d))]
print(f"Found crops: {crops}\n")

dataset_info = {}

for crop in crops:
    crop_path = os.path.join(DATASET_ROOT, crop)
    classes = sorted([d for d in os.listdir(crop_path) if os.path.isdir(os.path.join(crop_path, d))])

    crop_data = []
    print(f"--- {crop.capitalize()} ---")
    print(f"Classes: {classes}")

    for cls in classes:
        cls_path = os.path.join(crop_path, cls)
        images = glob.glob(os.path.join(cls_path, '*.*'))
        images = [img for img in images if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
        crop_data.append({'Class': cls, 'Count': len(images), 'Images': images})

    df = pd.DataFrame(crop_data)
    dataset_info[crop] = {'classes': classes, 'df': df}

    display(df[['Class', 'Count']])
    print(f"Total: {df['Count'].sum()}\n")

## 2. 3-Way Stratified Split (Train 70% / Validation 15% / Test 15%)

In [ ]:
splits = {}

for crop, info in dataset_info.items():
    df = info['df']
    classes = info['classes']

    all_paths = []
    all_labels = []

    for idx, row in df.iterrows():
        paths = row['Images']
        labels = [idx] * len(paths)
        all_paths.extend(paths)
        all_labels.extend(labels)

    # First split: 70% train, 30% temp (val+test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        all_paths, all_labels, test_size=0.3, random_state=SEED, stratify=all_labels
    )

    # Second split: 50/50 of temp -> 15% val, 15% test of total
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
    )

    splits[crop] = {
        'train_paths': X_train, 'train_labels': y_train,
        'val_paths': X_val, 'val_labels': y_val,
        'test_paths': X_test, 'test_labels': y_test,
        'num_classes': len(classes)
    }

    # Verify
    assert len(np.unique(y_train)) == len(classes), f"Missing class in {crop} train!"
    assert len(np.unique(y_val)) == len(classes), f"Missing class in {crop} val!"
    assert len(np.unique(y_test)) == len(classes), f"Missing class in {crop} test!"

    print(f"{crop.upper()} - Split Verified!\n"
          f"  Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}\n")

    for ci, cn in enumerate(classes):
        print(f"    {cn}: Train={y_train.count(ci)}, Val={y_val.count(ci)}, Test={y_test.count(ci)}")
    print()

NameError: name 'dataset_info' is not defined

## 3. Dataset Construction (tf.data.Dataset)

In [ ]:
def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def create_dataset(paths, labels, preprocess_func=None, is_training=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if is_training:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    if preprocess_func:
        def apply_preprocess(img, label):
            return preprocess_func(img), label
        ds = ds.map(apply_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

## 4. Model Building, Training & Evaluation Functions

In [ ]:
def build_model(base_model_fn, num_classes):
    base_model = base_model_fn(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
    base_model.trainable = False

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def train_and_evaluate(crop_name, model_name, base_model_fn, preprocess_func):
    print(f"\n{'='*60}")
    print(f"  {model_name} on {crop_name.upper()}")
    print(f"{'='*60}")

    sd = splits[crop_name]
    classes = dataset_info[crop_name]['classes']
    num_classes = sd['num_classes']

    train_ds = create_dataset(sd['train_paths'], sd['train_labels'], preprocess_func, True)
    val_ds = create_dataset(sd['val_paths'], sd['val_labels'], preprocess_func, False)
    test_ds = create_dataset(sd['test_paths'], sd['test_labels'], preprocess_func, False)

    model = build_model(base_model_fn, num_classes)

    checkpoint_path = os.path.join(OUTPUT_DIR, f'best_{crop_name}_{model_name}.keras')
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
    )

    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[checkpoint], verbose=1)

    # --- Plots ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title(f'{model_name} - {crop_name.upper()} Accuracy')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.legend()
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title(f'{model_name} - {crop_name.upper()} Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.legend()
    plt.tight_layout(); plt.show()

    # --- Evaluate on Validation Set ---
    print("\n--- Validation Set Evaluation ---")
    best_model = tf.keras.models.load_model(checkpoint_path)
    y_pred_val = np.argmax(best_model.predict(val_ds), axis=1)
    y_true_val = sd['val_labels']
    val_acc = accuracy_score(y_true_val, y_pred_val)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_true_val, y_pred_val, average='macro')
    print(f"Accuracy: {val_acc:.4f} | Precision: {p_m:.4f} | Recall: {r_m:.4f} | F1: {f1_m:.4f}")
    print(classification_report(y_true_val, y_pred_val, target_names=classes))

    # --- Evaluate on Test Set ---
    print("--- Test Set Evaluation ---")
    y_pred_test = np.argmax(best_model.predict(test_ds), axis=1)
    y_true_test = sd['test_labels']
    test_acc = accuracy_score(y_true_test, y_pred_test)
    p_m_t, r_m_t, f1_m_t, _ = precision_recall_fscore_support(y_true_test, y_pred_test, average='macro')
    print(f"Accuracy: {test_acc:.4f} | Precision: {p_m_t:.4f} | Recall: {r_m_t:.4f} | F1: {f1_m_t:.4f}")
    report = classification_report(y_true_test, y_pred_test, target_names=classes, output_dict=True)
    print(classification_report(y_true_test, y_pred_test, target_names=classes))

    # Save classification report
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(os.path.join(OUTPUT_DIR, f'report_{crop_name}_{model_name}.csv'))

    # Confusion Matrix (Test)
    cm = confusion_matrix(y_true_test, y_pred_test)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f'Confusion Matrix - {model_name} ({crop_name.upper()}) [Test]')
    plt.ylabel('True'); plt.xlabel('Predicted'); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'cm_{crop_name}_{model_name}.png'))
    plt.show()

    return {
        'Crop': crop_name, 'Model': model_name,
        'Val_Accuracy': val_acc, 'Val_F1': f1_m,
        'Test_Accuracy': test_acc, 'Test_F1': f1_m_t
    }

## 5. Run All Models on Both Crops

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mobilenet_preprocess

MODELS = {
    'ResNet50': (tf.keras.applications.ResNet50, resnet_preprocess),
    'EfficientNetB0': (tf.keras.applications.EfficientNetB0, effnet_preprocess),
    'MobileNetV3Large': (tf.keras.applications.MobileNetV3Large, mobilenet_preprocess)
}

all_results = []

for crop in ['guava', 'citrus']:
    for model_name, (base_fn, preprocess_fn) in MODELS.items():
        result = train_and_evaluate(crop, model_name, base_fn, preprocess_fn)
        all_results.append(result)

print("\nAll experiments completed!")

## 6. Final Comparison Table

In [ ]:
results_df = pd.DataFrame(all_results)
results_df = results_df[['Crop', 'Model', 'Val_Accuracy', 'Val_F1', 'Test_Accuracy', 'Test_F1']]
results_df = results_df.sort_values(by=['Crop', 'Test_Accuracy'], ascending=[True, False])

results_df.to_csv(os.path.join(OUTPUT_DIR, 'final_comparison.csv'), index=False)
display(results_df)

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for i, crop in enumerate(['guava', 'citrus']):
    crop_df = results_df[results_df['Crop'] == crop]
    x = np.arange(len(crop_df))
    w = 0.35
    axes[i].bar(x - w/2, crop_df['Val_Accuracy'], w, label='Val Accuracy')
    axes[i].bar(x + w/2, crop_df['Test_Accuracy'], w, label='Test Accuracy')
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(crop_df['Model'], rotation=15)
    axes[i].set_title(f'{crop.upper()} - Model Comparison')
    axes[i].set_ylabel('Accuracy')
    axes[i].legend()
    axes[i].set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'comparison_chart.png'))
plt.show()